<a href="https://colab.research.google.com/github/zainamri/kumpulantugas/blob/main/tgs4textmining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn nltk Sastrawi

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

In [ ]:
df = pd.read_csv('data_dilabeli.csv')
df = df.dropna(subset=['final_text', 'label'])
df['label'] = df['label'].astype(int)
df.head()

,Teks Komentar,final_text,label
0,"Rapatkan barisan semua suku di papua,untuk ber...",rapat baris suku papuauntuk tindak melawanjang...,1
1,Baru sadar bahwa pemerintah konoha itu penjaja...,sadar perintah konoha jajah pulau etnis sejaht...,0
2,"Salam Kenal KK,.. Saya senang sekali nonton vi...",salam kenal kk senang nonton video kk dorang l...,1
3,PAPUA BUKAN TANAH KOSONG!!,papua tanah kosong,1
4,https://youtu.be/Nf2OVkfLRLg?si=9Y1sJ-rbT3tYnIK3,httpsyoutubenfovkflrlgsiysjrbttynik,1


In [ ]:
# Menyeimbangkan data

jumlah_terkecil = df['label'].value_counts().min()
df_positif = df[df['label'] == 1]
df_negatif = df[df['label'] == 0]

# Memotong data mayoritas agar sama dengan data minoritas
if len(df_positif) > len(df_negatif):
    df_positif = df_positif.sample(jumlah_terkecil, random_state=42)
elif len(df_negatif) > len(df_positif):
    df_negatif = df_negatif.sample(jumlah_terkecil, random_state=42)

df_seimbang = pd.concat([df_positif, df_negatif])

print("DISTRIBUSI DATA SETELAH DISEIMBANGKAN")
print(df_seimbang['label'].value_counts())

X_teks = df_seimbang['final_text']
y_label = df_seimbang['label']

DISTRIBUSI DATA SETELAH DISEIMBANGKAN
label
1    31
0    31
Name: count, dtype: int64


In [ ]:
# BAG OF WORDS (BoW)
bow = CountVectorizer()
X_bow = bow.fit_transform(X_teks)
print("=== HASIL BAG OF WORDS (BoW) ===")
print("Sebagian Vocabulary BoW:", bow.get_feature_names_out()[:10])
print(X_bow.toarray()[:2]) # Tampilkan 2 baris awal saja sebagai contoh
print("======================\n")

=== HASIL BAG OF WORDS (BoW) ===
Sebagian Vocabulary BoW: ['abad' 'aborigin' 'acak' 'aceh' 'ada' 'adab' 'adat' 'adil' 'afrika'
 'agenda']
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]



In [ ]:
# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(X_teks)
print("=== HASIL TF-IDF ===")
print(X_tfidf.toarray()[:2]) # Tampilkan 2 baris awal saja sebagai contoh
print("==========================\n")

=== HASIL TF-IDF ===
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]



In [ ]:
# PELATIHAN MODEL NAIVE BAYES
# Pisahkan data belajar (80%) dan ujian (20%) menggunakan hasil TF-IDF
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_label, test_size=0.2, random_state=42)

# Latih mesin
model = MultinomialNB()
model.fit(X_train, y_train)

MultinomialNB()

In [ ]:
# EVALUASI AKURASI
y_tebakan = model.predict(X_test)
print(" RAPOR AKURASI NAIVE BAYES ")
print(classification_report(y_test, y_tebakan))
print("=================================\n")

 RAPOR AKURASI NAIVE BAYES 
              precision    recall  f1-score   support

           0       0.50      0.83      0.62         6
           1       0.67      0.29      0.40         7

    accuracy                           0.54        13
   macro avg       0.58      0.56      0.51        13
weighted avg       0.59      0.54      0.50        13




In [ ]:
# FUNGSI PREDIKSI KALIMAT BARU
# Siapkan alat pembersih teks baru
stop_words = set(stopwords.words('indonesian'))
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def tebak_sentimen(kalimat):
    # Proses pembersihan harus sama persis dengan saat preprocessing
    text = str(kalimat).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(w) for w in tokens]
    kalimat_final = " ".join(tokens)

    # Ubah teks ke angka lalu tebak
    vector_angka = tfidf.transform([kalimat_final])
    tebakan = model.predict(vector_angka)[0]

    hasil = "POSITIF (1)" if tebakan == 1 else "NEGATIF (0)"
    return f"Kalimat Asli   : '{kalimat}'\nDibaca Mesin   : '{kalimat_final}'\nHasil Tebakan  : {hasil}\n{'-'*60}"

# Uji coba kalimat komentar yang positif & negatif
print(" UJI COBA PREDIKSI ")
print(tebak_sentimen("Program ketahanan pangan ini sangat membantu masyarakat pedalaman."))
print(tebak_sentimen("Parah banget alam papua dihancurkan tanpa memikirkan masa depan."))

 UJI COBA PREDIKSI 
Kalimat Asli   : 'Program ketahanan pangan ini sangat membantu masyarakat pedalaman.'
Dibaca Mesin   : 'program tahan pangan bantu masyarakat dalam'
Hasil Tebakan  : POSITIF (1)
------------------------------------------------------------
Kalimat Asli   : 'Parah banget alam papua dihancurkan tanpa memikirkan masa depan.'
Dibaca Mesin   : 'parah banget alam papua hancur pikir'
Hasil Tebakan  : NEGATIF (0)
------------------------------------------------------------
